In [1]:
import pandas as pd
import numpy as np
import re

# -------------------------------------------------------------------
# Config: update these filenames if yours differ
# -------------------------------------------------------------------

SOLUTION_PATH = "/kaggle/input/benchrec-real-world-cash-reconciliation-dataset/BenchRec_cash_v1.0_solution.csv"
SUBMISSION_PATH= "/kaggle/working/submission_with_detail.csv"

# -------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------
def norm_id(x):
    """Normalize B_id; fixes cases like '12345.0' vs '12345'."""
    if x is None:
        return ""
    s = str(x).strip()
    if s.lower() == "nan" or s == "":
        return ""
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    return s

def parse_alloc_set(raw):
    """
    Parse targetAllocation into a set.
    Accepts:
      - single string: "ABC"
      - bracketed list: "[ABC,DEF]"
      - quoted variants: '"[ABC,DEF]"'
    """
    if raw is None:
        return set()
    s = str(raw).strip()
    if s == "" or s.lower() == "nan":
        return set()

    # remove surrounding quotes if present
    if len(s) >= 2 and ((s[0] == s[-1] == '"') or (s[0] == s[-1] == "'")):
        s = s[1:-1].strip()

    # bracket list
    if s.startswith("[") and s.endswith("]"):
        inner = s[1:-1].strip()
        if inner == "":
            return set()
        parts = [p.strip() for p in inner.split(",") if p.strip() != ""]
        parts = [p.strip('"').strip("'") for p in parts]
        return set(parts)

    # single value
    return {s}

# -------------------------------------------------------------------
# Load
# -------------------------------------------------------------------
sol = pd.read_csv(SOLUTION_PATH, dtype=str, on_bad_lines='skip', engine='python')
sub = pd.read_csv(SUBMISSION_PATH, dtype=str, on_bad_lines='skip', engine='python')

# Validate required columns (case-insensitive for solution)
sub_cols = {c.lower(): c for c in sub.columns}
sol_cols = {c.lower(): c for c in sol.columns}

required_sub = ["b_id", "targetallocation"]
required_sol = ["b_id", "targetallocation"]

for col in required_sub:
    if col not in sub_cols:
        raise ValueError(f"Submission missing column '{col}'. Found: {list(sub.columns)}")
for col in required_sol:
    if col not in sol_cols:
        raise ValueError(f"Solution missing column '{col}'. Found: {list(sol.columns)}")

sub_b   = sub_cols["b_id"]
sub_ta  = sub_cols["targetallocation"]
sol_b   = sol_cols["b_id"]
sol_ta  = sol_cols["targetallocation"]

# Normalize IDs
sub[sub_b] = sub[sub_b].map(norm_id)
sol[sol_b] = sol[sol_b].map(norm_id)

# Parse allocation sets
sub["pred_set"]  = sub[sub_ta].apply(parse_alloc_set)
sol["true_set"]  = sol[sol_ta].apply(parse_alloc_set)

# Merge on B_id
df = sol[[sol_b, sol_ta, "true_set"]].merge(
    sub[[sub_b, sub_ta, "pred_set"]],
    left_on=sol_b, right_on=sub_b, how="left", suffixes=('_true','_pred')
)

# Fill missing predictions
df[sub_ta + '_pred'] = df[sub_ta + '_pred'].fillna("")
df["pred_set"] = df[sub_ta + '_pred'].apply(parse_alloc_set)

# Metrics
df["is_matched"] = df["pred_set"].apply(lambda s: len(s) > 0)

# Correctness rule: predicted allocation is correct if pred_set ⊆ true_set (and pred_set non-empty)
# i.e. the predicted match  members are a subset of the labelled match members. (Sometime in manually created matches
# many transactions  are be grouped together in order to minimise gui operations)
df["subset_correct"] = df.apply(lambda r: (len(r["pred_set"]) > 0) and r["pred_set"].issubset(r["true_set"]), axis=1)


total = len(df)
matched = int(df["is_matched"].sum())
unmatched = total - matched

subset_ok = int(df["subset_correct"].sum())
matched_wrong_subset = int((df["is_matched"] & ~df["subset_correct"]).sum())

match_rate = matched / total if total else np.nan
subset_acc_matched_only = subset_ok / matched if matched else np.nan


#
# Define the STP precision requirement
# This is the precision needed for match decisions to be STP'd and maintain operational controls.
#
stpActualPrecisionTarget = 0.999
# This is the estimated label error rate in the dataset based off of manual eyeballing and review.
estimatedLabelErrorRate = 0.002
#
achievableStpprecisionTarget = (stpActualprecisionTarget - estimatedLabelErrorRate)
# Print results
print("---- BenchRec Evaluation ----")
print(f"STP precision Requirement: {achievableStpprecisionTarget:.2%}")
print(f"# Total B rows (solution): {total:,}")
print(f"# Matched (non-empty prediction): {matched:,}")
print(f"# Unmatched (no match prediction): {unmatched:,}")
print(f"precision of Match Predictions: {subset_acc_matched_only:.2%}")


if ( subset_acc_matched_only > achievableStpprecisionTarget ):
    print( "precision requirement met")
    print(f"Match rate: {match_rate:.4%}")
else:
    print( "precision requirement not met")


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/submission_with_detail.csv'